# E-Commerce Sales & Customer Analytics - Python EDA

This notebook performs exploratory data analysis for the e-commerce sales project. It covers data loading, cleaning, feature engineering, KPI calculation, visualization, and business insights.

## 1. Import Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
pd.set_option("display.max_columns", 50)

## 2. Load Dataset

In [ ]:
root = Path.cwd()
data_path = root / "Data & Resources" / "ECOMM DATA.xlsx"

if not data_path.exists():
    root = root.parent
    data_path = root / "Data & Resources" / "ECOMM DATA.xlsx"

orders = pd.read_excel(data_path, sheet_name="Orders")
returns = pd.read_excel(data_path, sheet_name="Returns")
people = pd.read_excel(data_path, sheet_name="People")

orders.shape, returns.shape, people.shape

## 3. Data Cleaning

In [ ]:
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace("-", "_", regex=False)
        .str.replace(" ", "_", regex=False)
    )
    return df


orders = clean_columns(orders)
returns = clean_columns(returns)
people = clean_columns(people)

text_columns = ["category", "sub_category", "segment", "market", "region", "ship_mode"]
for column in text_columns:
    orders[column] = orders[column].astype(str).str.strip().str.title()

orders["order_date"] = pd.to_datetime(orders["order_date"])
orders["ship_date"] = pd.to_datetime(orders["ship_date"])

null_summary = orders.isna().sum().sort_values(ascending=False)
duplicate_rows = orders.duplicated().sum()
duplicate_order_products = orders.duplicated(subset=["order_id", "product_id", "product_name"]).sum()
data_types = orders.dtypes

print("Top null counts:")
display(null_summary.head(10))
print(f"Duplicate full rows: {duplicate_rows:,}")
print(f"Duplicate order/product rows: {duplicate_order_products:,}")
display(data_types)

### Outlier and Category Validation

Outliers are flagged using the IQR method. They are reviewed rather than automatically removed because unusually high sales or shipping values may be valid business transactions.

In [ ]:
numeric_columns = ["sales", "quantity", "discount", "shipping_cost", "profit"]
outlier_summary = []

for column in numeric_columns:
    q1 = orders[column].quantile(0.25)
    q3 = orders[column].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = orders[(orders[column] < lower) | (orders[column] > upper)]
    outlier_summary.append({
        "column": column,
        "lower_bound": lower,
        "upper_bound": upper,
        "potential_outliers": len(outliers)
    })

display(pd.DataFrame(outlier_summary))
orders[["category", "segment", "market", "ship_mode"]].nunique()

## 4. Feature Engineering

New fields are created for trend analysis, shipping performance, return tracking, discount bands, and profitability analysis.

In [ ]:
orders["order_year"] = orders["order_date"].dt.year
orders["order_month"] = orders["order_date"].dt.month
orders["order_month_name"] = orders["order_date"].dt.strftime("%b")
orders["year_month"] = orders["order_date"].dt.to_period("M").astype(str)
orders["shipping_delay"] = (orders["ship_date"] - orders["order_date"]).dt.days
orders["profit_margin"] = orders["profit"] / orders["sales"]
orders["customer_segment"] = orders["segment"]
orders["sales_bucket"] = pd.cut(
    orders["sales"],
    bins=[-0.01, 100, 500, 1000, float("inf")],
    labels=["Low", "Medium", "High", "Premium"]
)

returned_orders = set(returns["order_id"].dropna())
orders["is_returned"] = orders["order_id"].isin(returned_orders)

orders["discount_band"] = pd.cut(
    orders["discount"],
    bins=[-0.01, 0, 0.10, 0.20, 0.30, 1],
    labels=["No Discount", "0-10%", "10-20%", "20-30%", "30%+"]
)

orders.head()

## 5. KPI Calculations

In [ ]:
kpis = pd.Series({
    "Total Sales": orders["sales"].sum(),
    "Total Profit": orders["profit"].sum(),
    "Total Quantity": orders["quantity"].sum(),
    "Total Shipping Cost": orders["shipping_cost"].sum(),
    "Distinct Orders": orders["order_id"].nunique(),
    "Distinct Customers": orders["customer_id"].nunique(),
    "Profit Margin": orders["profit"].sum() / orders["sales"].sum(),
    "Return Rate": orders["is_returned"].mean()
})

kpis

## 6. Sales Trend

In [ ]:
monthly_sales = (
    orders.groupby("year_month", as_index=False)
    .agg(sales=("sales", "sum"), profit=("profit", "sum"))
)

sns.lineplot(data=monthly_sales, x="year_month", y="sales", marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

**Insight:** Monthly sales show how demand changes over time and help identify growth patterns, seasonality, and periods that may need deeper investigation.

## 7. Category and Sub-Category Performance

In [ ]:
category_perf = (
    orders.groupby("category", as_index=False)
    .agg(sales=("sales", "sum"), profit=("profit", "sum"), quantity=("quantity", "sum"))
    .sort_values("sales", ascending=False)
)

sns.barplot(data=category_perf, x="category", y="sales", hue="category", legend=False)
plt.title("Sales by Category")
plt.xlabel("Category")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()

category_perf

In [ ]:
subcategory_profit = (
    orders.groupby(["category", "sub_category"], as_index=False)
    .agg(sales=("sales", "sum"), profit=("profit", "sum"))
    .sort_values("profit", ascending=False)
)

sns.barplot(data=subcategory_profit.head(10), y="sub_category", x="profit", hue="category")
plt.title("Top 10 Sub-Categories by Profit")
plt.xlabel("Profit")
plt.ylabel("Sub-Category")
plt.tight_layout()
plt.show()

subcategory_profit.head(10)

**Insight:** Category-level sales should be compared with profit, because high revenue does not always mean strong profitability.

## 8. Market and Regional Performance

In [ ]:
market_perf = (
    orders.groupby("market", as_index=False)
    .agg(sales=("sales", "sum"), profit=("profit", "sum"), orders=("order_id", "nunique"))
    .assign(profit_margin=lambda df: df["profit"] / df["sales"])
    .sort_values("sales", ascending=False)
)

sns.barplot(data=market_perf, x="market", y="sales", hue="market", legend=False)
plt.title("Sales by Market")
plt.xlabel("Market")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()

market_perf

In [ ]:
top_countries = (
    orders.groupby("country", as_index=False)
    .agg(sales=("sales", "sum"), profit=("profit", "sum"))
    .sort_values("sales", ascending=False)
    .head(10)
)

sns.barplot(data=top_countries, y="country", x="sales", hue="country", legend=False)
plt.title("Top 10 Countries by Sales")
plt.xlabel("Sales")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

top_countries

**Insight:** Market and country performance highlights where the business should prioritize expansion, retention, and operational support.

## 9. Customer Segment Analysis

In [ ]:
segment_perf = (
    orders.groupby("customer_segment", as_index=False)
    .agg(
        customers=("customer_id", "nunique"),
        orders=("order_id", "nunique"),
        sales=("sales", "sum"),
        profit=("profit", "sum")
    )
    .assign(profit_margin=lambda df: df["profit"] / df["sales"])
    .sort_values("sales", ascending=False)
)

sns.barplot(data=segment_perf, x="customer_segment", y="sales", hue="customer_segment", legend=False)
plt.title("Sales by Customer Segment")
plt.xlabel("Customer Segment")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()

segment_perf

**Insight:** Segment analysis shows which customer groups are most valuable and where targeted campaigns may improve revenue or margin.

## 10. Sales Bucket Analysis

In [ ]:
sales_bucket_perf = (
    orders.groupby("sales_bucket", observed=False, as_index=False)
    .agg(line_items=("order_id", "count"), sales=("sales", "sum"), profit=("profit", "sum"))
    .assign(profit_margin=lambda df: df["profit"] / df["sales"])
)

sns.barplot(data=sales_bucket_perf, x="sales_bucket", y="profit_margin", hue="sales_bucket", legend=False)
plt.title("Profit Margin by Sales Bucket")
plt.xlabel("Sales Bucket")
plt.ylabel("Profit Margin")
plt.tight_layout()
plt.show()

sales_bucket_perf

**Insight:** Sales buckets help compare smaller transactions with high-value orders and show whether larger transactions are also more profitable.

## 11. Discount Impact on Profitability

In [ ]:
discount_perf = (
    orders.groupby("discount_band", observed=False, as_index=False)
    .agg(line_items=("order_id", "count"), sales=("sales", "sum"), profit=("profit", "sum"))
    .assign(profit_margin=lambda df: df["profit"] / df["sales"])
)

sns.barplot(data=discount_perf, x="discount_band", y="profit_margin", hue="discount_band", legend=False)
plt.title("Profit Margin by Discount Band")
plt.xlabel("Discount Band")
plt.ylabel("Profit Margin")
plt.tight_layout()
plt.show()

discount_perf

**Insight:** Discount bands reveal whether higher discounts are helping sales or damaging profitability. This supports a more controlled promotional strategy.

## 12. Shipping and Return Analysis

In [ ]:
ship_perf = (
    orders.groupby("ship_mode", as_index=False)
    .agg(
        orders=("order_id", "nunique"),
        sales=("sales", "sum"),
        shipping_cost=("shipping_cost", "sum"),
        avg_shipping_delay=("shipping_delay", "mean")
    )
    .assign(shipping_cost_ratio=lambda df: df["shipping_cost"] / df["sales"])
    .sort_values("sales", ascending=False)
)

sns.barplot(data=ship_perf, x="ship_mode", y="shipping_cost_ratio", hue="ship_mode", legend=False)
plt.title("Shipping Cost Ratio by Ship Mode")
plt.xlabel("Ship Mode")
plt.ylabel("Shipping Cost / Sales")
plt.tight_layout()
plt.show()

ship_perf

In [ ]:
return_by_market = (
    orders.groupby("market", as_index=False)
    .agg(total_orders=("order_id", "nunique"), returned_orders=("is_returned", "sum"))
    .assign(return_rate=lambda df: df["returned_orders"] / df["total_orders"])
    .sort_values("return_rate", ascending=False)
)

return_by_market

**Insight:** Shipping mode and return-rate analysis can help identify operational areas that affect customer satisfaction and margin.

## 13. Final Business Recommendations

- Prioritize high-sales markets while monitoring profit margin, not just revenue.
- Review sub-categories with high sales but weak or negative profit.
- Set discount guardrails for products and regions where discounts reduce margin.
- Track returns by market and category to identify quality or fulfillment issues.
- Use customer segment performance to design targeted marketing and retention campaigns.
- Combine growth goals with profit-margin guardrails so expansion improves both revenue and business value.